# Fear-Gauge — a quantitative teardown 🔬
### Random-day null · price-drop control · block bootstrap · window selection · martingale risk-of-ruin

![Signal: Mixed](https://img.shields.io/badge/Signal-Mixed-dab617?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Double--down: Ruin--prone](https://img.shields.io/badge/Double--down-Ruin--prone-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim now carrying its standard error.* The thesis we're testing: the post-spike rebound is **statistically real** (it's the variance risk premium) but **uninvestable** (short-skew premium + a martingale that ignores 2008).

> ⚠️ **Not investment advice.** Reproducible research; fixed seeds. Data: ^GSPC, SPY, ^VIX from Yahoo! Finance (first run hits the network, later runs use the local cache under `../_cache/`). Methods: event study, random-day permutation null, label-permutation cross-study control, block bootstrap, window-selection test, martingale risk-of-ruin — see [`docs/references.md`](../docs/references.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back into intuition — so this notebook still reads even if you skim the maths. House style in [METHODOLOGY.md](../../../METHODOLOGY.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # study root (fear_gauge/ lives there)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9.5, 5.2)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
from fear_gauge import data, triggers, exits, eventstudy, benchmark, backtest, robustness

# The headline pair: S&P 500 spot for stats (deep history), aligned with the VIX.
spx, vix = data.aligned("^GSPC")
print(f"{len(spx):,} common sessions  {spx.index.min().date()} -> {spx.index.max().date()}")


9,174 common sessions  1990-01-02 -> 2026-06-05


In [2]:
# Both faces of the market + the gauge. Spot for stats, ETF for the cost layer.
spy, vix_spy = data.aligned('SPY')
for nm, (mkt, vx) in {'^GSPC': (spx, vix), 'SPY': (spy, vix_spy)}.items():
    print(f"{nm:6s} {len(mkt):>6,} sessions  {mkt.index.min().date()} -> {mkt.index.max().date()}")

^GSPC   9,174 sessions  1990-01-02 -> 2026-06-05
SPY     8,395 sessions  1993-01-29 -> 2026-06-05


## 1 · The claim, as testable hypotheses

H₁ (level): E[r_{t→t+h} | VIX_t ≥ 30] > E[r_{t→t+h}] with HAC-robust t > 2.
H₁ (spike): same, conditioned on ΔVIX_t ≥ +30%.
H₀: excess ≈ 0 — the forward return is drift + beta. Horizons h ∈ {1, 5, 21} to match the chart's +1d/+1w/+1m exactly.

In [3]:
FAMILY = {
    'V1_level_30': triggers.level(vix, 30),
    'V1_level_40': triggers.level(vix, 40),
    'V1_level_50': triggers.level(vix, 50),
    'V2_spike':    triggers.spike(vix),
    'V3_low_base': triggers.spike_from_base(vix, base_max=20),
    'V3_high_base':triggers.spike_from_base(vix, base_min=30),
}
events = {k: triggers.first_crossings(v, cooldown=21) for k, v in FAMILY.items()}
pd.Series({k: int(v.sum()) for k, v in events.items()}, name='fresh events').to_frame()

,fresh events
V1_level_30,54
V1_level_40,17
V1_level_50,6
V2_spike,36
V3_low_base,27
V3_high_base,5


## 3–4 · The random-day null, across the whole family

`p_greater` = P(a random basket of the same size beats the conditional mean). Small ⇒ the gauge adds something beyond drift.

In [4]:
rows = []
for name, sig in events.items():
    t = benchmark.conditional_vs_unconditional(spx, sig, horizons=(1, 5, 21), n_iter=2000)
    for h, r in t.iterrows():
        rows.append({'trigger': name, 'h': h, 'n': int(r.n_events),
                     'excess': r.excess, 'p_greater': r.p_greater})
null_tbl = pd.DataFrame(rows).set_index(['trigger', 'h'])
null_tbl

n  excess  p_greater
trigger      h                        
V1_level_30  1   54  0.0009     0.2680
             5   54  0.0102     0.0000
             21  54  0.0132     0.0115
V1_level_40  1   17  0.0096     0.0005
             5   17  0.0168     0.0005
             21  17  0.0281     0.0035
V1_level_50  1    6  0.0137     0.0060
             5    6 -0.0478     1.0000
             21   6 -0.0088     0.7060
V2_spike     1   35  0.0013     0.2535
             5   35  0.0031     0.2145
             21  35 -0.0002     0.5070
V3_low_base  1   26 -0.0002     0.5250
             5   26  0.0040     0.1865
             21  26 -0.0003     0.5040
V3_high_base 1    5  0.0369     0.0000
             5    5  0.0032     0.3795
             21   5  0.0310     0.0605

## 4 · The cross-study control — does VIX beat the price drop?

The confound that makes this study hard: VIX-high ≈ price-low. We pit each VIX trigger against Study 02's T1 (a −3% close) on the *same* forward returns, with a label-permutation test on the gap. **A VIX trigger that clears the random-day null but not this control is the falling knife in vol coordinates.**

In [5]:
t1 = triggers.first_crossings(spx['r_cc'] <= -0.03, cooldown=21)
ctrl = {}
for name in ['V1_level_30', 'V2_spike', 'V3_high_base']:
    ctrl[name] = benchmark.excess_vs_alternative(spx, events[name], t1, horizons=(1, 5, 21))
pd.concat(ctrl, names=['trigger', 'h'])[['mean_signal', 'mean_alt', 'gap', 'p_signal_gt_alt']]

mean_signal  mean_alt     gap  p_signal_gt_alt
trigger      h                                                 
V1_level_30  1        0.0013    0.0026 -0.0013           0.6205
             5        0.0122    0.0031  0.0091           0.1290
             21       0.0213    0.0086  0.0127           0.1990
V2_spike     1        0.0017    0.0026 -0.0009           0.5725
             5        0.0050    0.0031  0.0019           0.4015
             21       0.0079    0.0086 -0.0007           0.5005
V3_high_base 1        0.0373    0.0026  0.0347           0.0035
             5        0.0051    0.0031  0.0020           0.4605
             21       0.0391    0.0086  0.0305           0.2080

## 4 · Clustering and selection — the two ways the chart cheats

**(a) Clustering.** 4 of the chart's 23 events are Feb–Mar 2020. The iid permutation overstates significance; a block bootstrap that resamples contiguous months gives an honest CI on the excess.

In [6]:
boot = {name: robustness.block_bootstrap_excess(spx, events[name], horizon=21,
                                                 block=21, n_iter=2000)
        for name in ['V1_level_30', 'V2_spike', 'V3_high_base']}
pd.DataFrame(boot).T[['mean', 'ci_low', 'ci_high', 'p_excess_le_0']]

,mean,ci_low,ci_high,p_excess_le_0
V1_level_30,0.0131,-0.0024,0.0273,0.0460
V2_spike,0.0000,-0.0211,0.0174,0.4645
V3_high_base,0.0311,-0.0733,0.0992,0.2286


**(b) Selection.** The viral chart's window (2016–2026) excludes 2008. Recompute the same excess on the cherry-picked window vs the full history — if it shrinks or flips, the edge was a property of the *window*, not the *signal*.

In [7]:
robustness.window_sensitivity(spx, events['V2_spike'], horizon=21)

,start,end,n_events,mean_cond,mean_uncond,excess
window,,,,,,
chart (2016–2026),2016-06-01,2026-06-30,16,0.0170,0.0119,0.0051
full history,1990-01-02,2026-06-05,35,0.0079,0.0081,-0.0002
ex-2016+ (pre-window),1990-01-02,2016-05-31,19,0.0003,0.0067,-0.0064


## 5 · The verdict, with the numbers

Collate the decisive figures: random-day `p_greater`, the price-drop `gap` and its p-value, the block-bootstrap CI, and the window swing. Fill the README's beat-5 stamps from this cell once it's run on live data.

Expected shape: **Signal `REAL`** at short horizons (vol mean-reversion is robust) but **Tradability `MIRAGE`/`FRAGILE`** — the excess is VRP, it doesn't clearly beat the price drop, and it's carried by a handful of clustered crises.

## 6 · Could you trade it — the martingale's risk of ruin

Spot VIX isn't investable; the buy side is SPY (or a vol product with roll/decay). The decisive risk is the sizing rule: **"double down at 50"** adds capital as the tail fattens. We run it through history and report the probability it hits a ruin drawdown before recovering — on the full sample, and on the no-2008 window that sells it.

**First, trade the level honestly.** Buy SPY at the VIX≥30 cross, hold a month, charge realistic costs (including entry slippage into the spike). The Sharpe is modest and you sit in cash ~88% of the time — market-timing that underperforms buy-and-hold, not an edge.

In [8]:
sig30 = triggers.first_crossings(triggers.level(vix, 30), cooldown=21)
res = backtest.run(spx, sig30, exits.ExitRule(max_hold=21), backtest.CostModel())
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in res.stats.items()
       if k in ('n_trades','cagr','sharpe','max_drawdown','win_rate','exposure')})
backtest.cost_sweep(spx, sig30, exits.ExitRule(max_hold=21))  # entry-slippage sweep

{'n_trades': 51, 'cagr': 0.0225, 'sharpe': 0.7879, 'max_drawdown': -0.2852, 'win_rate': 0.7255, 'exposure': 0.1167}


,cagr,sharpe,total_return,max_drawdown,n_trades
panic_slippage_bps,,,,,
0,0.0232,0.8080,1.3046,-0.2852,51
5,0.0225,0.7879,1.2466,-0.2852,51
10,0.0218,0.7677,1.1900,-0.2852,51
20,0.0203,0.7274,1.0810,-0.2852,51
40,0.0175,0.6464,0.8787,-0.2990,51


**Then watch the family scan lie.** Scan every (trigger × exit) and the best cell posts an absurd Sharpe — on a handful of trades. Deflate it for the number of configs tried and it collapses back into noise. *Never read the best row.*

In [9]:
fam = {'V1_30': triggers.level(vix,30), 'V1_50': triggers.level(vix,50),
       'V2_spike': triggers.spike(vix), 'V3_high': triggers.spike_from_base(vix, base_min=30)}
scan = backtest.family_scan(spx, fam, exits.default_grid())
top = scan.iloc[0]
dsr = robustness.deflated_sharpe(top.sharpe, n_trials=len(scan), n_obs=int(top.avg_hold_days*top.n_trades))
print(f"best cell: {top.trigger} / {top.exit}  Sharpe={top.sharpe:.1f} on {int(top.n_trades)} trades")
print(f"deflated Sharpe (P it isn't luck over {len(scan)} configs): {dsr:.3f}")
scan.head(5)

best cell: V3_high / hold<=1d  Sharpe=13.9 on 5 trades
deflated Sharpe (P it isn't luck over 52 configs): 0.367


,trigger,exit,n_trades,cagr,sharpe,total_return,max_drawdown,win_rate,avg_hold_days
0,V3_high,hold<=1d,5,0.0048,13.8565,0.1911,-0.0068,0.6000,1.0000
1,V3_high,hold<=42d tp+3% sl-5%,5,0.0017,5.5829,0.0640,-0.0510,0.8000,1.2000
2,V3_high,hold<=21d tp+3% sl-5%,5,0.0017,5.5829,0.0640,-0.0510,0.8000,1.2000
3,V3_high,hold<=10d tp+3% sl-5%,5,0.0017,5.5829,0.0640,-0.0510,0.8000,1.2000
4,V3_high,hold<=42d tp+5% sl-10%,5,0.0066,5.5228,0.2701,-0.0709,1.0000,5.2000


**Now the martingale.** "Double down at 50", held a fixed quarter, tracking the deepest drawdown along the way — on the full sample versus the no-2008 window that sells the rule.

In [10]:
full = robustness.martingale_ruin(spx, vix, rung1=30, rung2=50)
win  = spx.index >= '2016-06-01'
windowed = robustness.martingale_ruin(spx[win], vix[win], rung1=30, rung2=50)
pd.DataFrame({'full history': full, 'chart window (2016–2026)': windowed})

,full history,chart window (2016–2026)
n_episodes,32.0000,11.0000
p_ruin,0.0625,0.0909
p_doubled,0.0938,0.1818
mean_terminal,0.0525,0.0777
worst_drawdown,-0.3286,-0.2171
worst_terminal,-0.2150,-0.0363


## 7 · Going further

- **VRP-hedged return** as the dependent variable — does any excess survive once you strip the variance premium? If not, case closed.
- **Term-structure conditioning** (VIX vs VIX3M backwardation at the event).
- **Deflated Sharpe** over the full (trigger × horizon × exit) grid, à la Study 02, for the backtest layer when `backtest.py` lands.
- **Cross-asset replication** on MOVE (bonds) and DVOL (crypto).

Engine: [`../../../quantlab/`](../../../quantlab/). Method: [`METHODOLOGY.md`](../../../METHODOLOGY.md).